# RSNA Knee — training on Colab (A100)

1. Runtime → Change runtime type → **A100 GPU** (High-RAM if offered).
2. Left sidebar → 🔑 Secrets → add `KAGGLE_API_TOKEN` = contents of `~/.kaggle/access_token`, enable *Notebook access*.
3. Run the cells top to bottom. Checkpoints live on Google Drive (`MyDrive/rsna-knee/`), so after a disconnect just run the same cells again: training resumes from the last finished epoch.

In [ ]:
from google.colab import userdata, drive
import os
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc; free -g | head -2; df -h /content | tail -1

In [ ]:
%%bash
set -e
mkdir -p ~/.kaggle && printf '%s' "$KAGGLE_API_TOKEN" > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
pip install -q -U 'kaggle>=2.0'
rm -rf /content/code && mkdir -p /content/code
kaggle datasets download kirillkholkin/rsna-knee-code -p /content/code --unzip -q
cd /content/code && ls && python kaggle/launch.py whoami

### Run 1 — EfficientNetV2-S, Raptor-style recipe (≈2–3 h on A100)
288 px, 30 windows over all series, 16 epochs, all non-gold studies, two seeds (`--folds 0 1`), top-3 weight averaging, labels = blend v2. First run downloads ~19 GB of cache from Kaggle (5–10 min).

In [ ]:
%%shell
cd /content/code
python colab/colab_train.py --name v3b-effv2s --folds 0 1 \
  --config configs/v3_raptor_effv2s.yaml --label-table configs/labels/label_table_blend_v2.parquet \
  --override optim.batch_size=4 --override optim.accum=2 --override model.grad_checkpoint=false --override data.num_workers=10 \
  --max-hours 11 --upload


### Run 2 — CoAtNet RMLP-2 384 (≈5–6 h on A100)
Same recipe on the backbone the strongest public single models use. Run after Run 1 if compute units allow.

In [ ]:
%%shell
cd /content/code
python colab/colab_train.py --name v3b-coatnet --folds 0 1 \
  --config configs/v3_raptor_coatnet.yaml --label-table configs/labels/label_table_blend_v2.parquet \
  --override optim.batch_size=2 --override optim.accum=4 --override data.num_workers=10 \
  --max-hours 11 --upload --skip-pip


### Ablation — same recipe without the transformers (≈1 h on A100)
Only the aggregator changes: per-finding attention pooling over windows, no series/study transformer. Compare `gold_macro_auc` per epoch with Run 1.

In [ ]:
%%shell
cd /content/code
python colab/colab_train.py --name v3c-simplehead --folds 0 \
  --config configs/v3c_simplehead_effv2s.yaml --label-table configs/labels/label_table_blend_v2.parquet \
  --override optim.batch_size=4 --override optim.accum=2 --override model.grad_checkpoint=false --override data.num_workers=10 \
  --max-hours 11 --upload --skip-pip


### Upload weights after interrupting a training cell
If a training cell was stopped with ■ (e.g. after the first seed finished), publish the finished folds from Drive:

In [ ]:
%%shell
cd /content/code
python colab/upload_runs.py --name v3-effv2s


### Label ablation under the holdout protocol (≈1 h for both)
Same model and recipe; val = fold 3 (941 studies) scored against a FIXED consensus reference table + all 58 gold (never trained on). Compare the `confident` and `gold` numbers of the two runs epoch by epoch; differences below 0.003 (`confident`) are noise.

In [ ]:
%%shell
cd /content && rm -rf code && mkdir code && kaggle datasets download kirillkholkin/rsna-knee-code -p /content/code --unzip -q
cd /content/code
python colab/colab_train.py --name v4a-blend3 --folds 3 --label-table configs/labels/label_table_blend_v3.parquet \
  --config configs/v4_base.yaml --eval-label-table configs/labels/label_table_ref_v1.parquet \
  --override optim.batch_size=4 --override optim.accum=2 --override model.grad_checkpoint=false --override data.num_workers=10 \
  --max-hours 11 --skip-pip
python colab/colab_train.py --name v4b-graded4 --folds 3 --label-table configs/labels/label_table_graded_v4.parquet \
  --config configs/v4_base.yaml --eval-label-table configs/labels/label_table_ref_v1.parquet \
  --override optim.batch_size=4 --override optim.accum=2 --override model.grad_checkpoint=false --override data.num_workers=10 \
  --max-hours 11 --skip-pip


### Result
`MyDrive/rsna-knee/work-<name>/runs/<name>/fold{0,1}/` holds `best.pt` (top-3 averaged weights), `config.yaml`, `priors.json`, `history.jsonl`. With `--upload` the same folder is published as the private Kaggle dataset `kirillkholkin/rsna-knee-runs-<name>`; tell Claude the run finished and it will attach the dataset to the inference kernel and submit.

Progress while a cell is running: open the Files panel → `drive/MyDrive/rsna-knee/work-<name>/train.log` (live log) and `…/runs/<name>/fold0/history.jsonl` (one line per finished epoch).